# 04 — Full Pipeline Run

This notebook runs the complete neoantigen prediction pipeline end-to-end on the HCC1395 breast cancer cell line data.

**Pipeline steps:**
1. Load HLA types from OptiType output
2. Parse missense variants from VEP-annotated VCF
3. Filter by gene expression
4. Load reference proteome
5. Generate peptide candidates
6. MHC-I binding/presentation prediction (MHCflurry)
7. Compute agretopicity scores
8. Rank by composite score

Data: `data/HCC1395_inputs/`

> **Note**: MHCflurry models must be downloaded before running step 6.  
> Run `mhcflurry-downloads fetch` once in your terminal if you haven't already.

In [5]:
import sys
import time
import pprint
import dataclasses
from pathlib import Path

import pandas as pd

# Add src to path if running from notebooks directory
sys.path.insert(0, str(Path("../src").resolve()))

from neoantigen_pipeline import NeoantigenPipeline
from neoantigen_pipeline.config import PipelineConfig

print("Imports OK")

Imports OK


In [6]:
# Define all data paths
DATA_DIR      = Path("../data/HCC1395_inputs")
VCF_PATH      = DATA_DIR / "annotated.expression.vcf.gz"
HLA_PATH      = DATA_DIR / "optitype_normal_result.tsv"
PROTEOME_PATH = DATA_DIR / "Homo_sapiens.GRCh38.pep.all.fa.gz"
CONFIG_PATH   = Path("../configs/default.yaml")

print("Path availability:")
for name, p in [("VCF", VCF_PATH), ("HLA", HLA_PATH),
                ("Proteome", PROTEOME_PATH), ("Config", CONFIG_PATH)]:
    print(f"  {name:10s}: {p.exists()}  ({p.resolve()})")

Path availability:
  VCF       : True  (/home/jan/Dropbox/personal_projects/Neoantigen_project/NeoantigenPipeline/data/HCC1395_inputs/annotated.expression.vcf.gz)
  HLA       : True  (/home/jan/Dropbox/personal_projects/Neoantigen_project/NeoantigenPipeline/data/HCC1395_inputs/optitype_normal_result.tsv)
  Proteome  : True  (/home/jan/Dropbox/personal_projects/Neoantigen_project/NeoantigenPipeline/data/HCC1395_inputs/Homo_sapiens.GRCh38.pep.all.fa.gz)
  Config    : True  (/home/jan/Dropbox/personal_projects/Neoantigen_project/NeoantigenPipeline/configs/default.yaml)


## Pipeline Configuration

In [7]:
# Load pipeline configuration from YAML
config = PipelineConfig.from_yaml(str(CONFIG_PATH))

print("PipelineConfig:")
try:
    pprint.pprint(dataclasses.asdict(config), indent=2)
except TypeError:
    # Fallback if config is not a dataclass
    pprint.pprint(vars(config) if hasattr(config, "__dict__") else str(config), indent=2)

PipelineConfig:
{ 'expression_filter': { 'expression_field': 'CSQ',
                         'filter_missing': False,
                         'min_expression': 1.0},
  'mhc_i': { 'alleles': ( 'HLA-A*29:02',
                          'HLA-B*45:01',
                          'HLA-B*82:02',
                          'HLA-C*06:02'),
             'binding_affinity_threshold_nm': 500.0,
             'peptide_lengths': (8, 9, 10, 11),
             'percentile_rank_threshold': 2.0,
             'use_presentation_score': True},
  'output_dir': 'results',
  'peptide_generation': { 'c_flank_length': 10,
                          'n_flank_length': 10,
                          'peptide_lengths': (8, 9, 10, 11)},
  'scoring': { 'agretopicity_weight': 0.2,
               'expression_weight': 0.2,
               'presentation_score_weight': 0.4,
               'vaf_weight': 0.2}}


## Running the Pipeline

The `NeoantigenPipeline.run()` method executes all steps in sequence.  
MHCflurry prediction is the most time-consuming step.

In [8]:
pipeline = NeoantigenPipeline(config)

print("Starting pipeline run ...")
t0 = time.time()
results = pipeline.run(
    str(VCF_PATH),
    str(HLA_PATH),
    str(PROTEOME_PATH),
)
elapsed = time.time() - t0

print(f"Pipeline completed in {elapsed:.1f}s")

Starting pipeline run ...
Predicting processing.


  0%|          | 0/1 [00:00<?, ?it/s]

4/4 [==============================] - 1s 217ms/step


100%|██████████| 1/1 [00:07<00:00,  7.88s/it]


Predicting affinities.


  0%|          | 0/4 [00:00<?, ?it/s]WARNING:root:4 peptides have nonstandard amino acids: <StringArray>
['XLALDAPQ', 'XLALDAPQH', 'XLALDAPQHS', 'XLALDAPQHSR']
Length: 4, dtype: str
  0%|          | 0/4 [00:00<?, ?it/s]


PredictionError: MHCflurry prediction failed: 4 peptides have nonstandard amino acids: <StringArray>
['XLALDAPQ', 'XLALDAPQH', 'XLALDAPQHS', 'XLALDAPQHSR']
Length: 4, dtype: str

## Results Overview

In [ ]:
# Convert results to DataFrame for inspection
df = results.to_dataframe()
print(f"Total neoantigen candidates: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
display(df.head(20))

In [ ]:
# Show top 10 candidates with key scoring columns
key_cols = ["gene", "mutation", "peptide", "best_allele",
            "presentation_score", "agretopicity",
            "composite_score", "composite_rank"]

# Use only columns that exist in the dataframe
available_cols = [c for c in key_cols if c in df.columns]
missing_cols   = [c for c in key_cols if c not in df.columns]

if missing_cols:
    print(f"Note: these expected columns were not found and will be skipped: {missing_cols}")
    print(f"Available columns: {list(df.columns)}")

print("\nTop 10 neoantigen candidates:")
display(df[available_cols].head(10))

## Saving Results

In [ ]:
import os

results_dir = Path("../results")
os.makedirs(str(results_dir), exist_ok=True)

out_path = results_dir / "HCC1395_neoantigens.tsv"
results.to_csv(str(out_path))

print(f"Results saved to {out_path.resolve()}")
print(f"File size: {out_path.stat().st_size / 1024:.1f} KB")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")